# FARUMASI shift dispatcher

Use a normal **CPU** runtime. Do not request a T4. The pharmacy step is Python and the network is small, so a GPU waits on the environment and the free T4 quota runs out before the results can be saved.

The patient walks a Kigali map. Licensed pharmacies can be empty or expensive. Unlicensed stalls are cheap and harmful. FARUMASI is the action that reveals live stock. The report comparison is still 10 settings for DQN, REINFORCE, PPO, and A2C. Each setting trains for 20,000 frames. On a Colab CPU that is about two to three hours. Checkpoints from the old dispatcher are ignored.

1. **Runtime → Run all**.
2. Allow the Google Drive prompt. After every finished setting the run is copied to `MyDrive/farumasi_rl`.
3. If Colab disconnects, open this notebook again and choose **Run all**. Finished settings are restored from Drive and skipped.
4. The last cell downloads `farumasi_rl_results.zip`. Keep that file.

The training cell prints each setting as it starts, then the frame, loss, and held-out score.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ["SDL_VIDEODRIVER"] = "dummy"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["MPLBACKEND"] = "Agg"
os.environ["PYTHONUNBUFFERED"] = "1"

import torch

print("python", sys.version.split()[0], "torch", torch.__version__, flush=True)
if sys.version_info < (3, 12):
    raise RuntimeError("TorchRL 0.15 needs Python 3.12 or newer. This runtime is " + sys.version.split()[0])

REPO = "https://github.com/Emmanuel-NS/Rl_Mission-based_Summative.git"
base = Path("/content") if Path("/content").exists() else Path.cwd()
dest = base / "Rl_Mission-based_Summative"
if not (dest / "main.py").exists():
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.check_call(["git", "clone", "--depth", "1", REPO, str(dest)])
os.chdir(dest)
print("working directory", Path.cwd(), flush=True)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    backup = Path("/content/drive/MyDrive/farumasi_rl")
    backup.mkdir(parents=True, exist_ok=True)
    os.environ["FARUMASI_BACKUP"] = str(backup)
    for name in ("logs", "models"):
        source = backup / name
        if source.exists():
            shutil.copytree(source, dest / name, dirs_exist_ok=True)
            print("restored", name, flush=True)
except Exception as error:
    print("Drive is not mounted, so a disconnect can wipe this session.", error, flush=True)

subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "gymnasium>=1.0", "pygame>=2.6", "cloudpickle", "packaging", "pyvers",
    ]
)
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "--no-deps",
        "torchrl==0.15.0", "tensordict==0.15.6", "hoptorch==0.1.4",
    ]
)

check = """
import torchrl
from torchrl.collectors import Collector
from torchrl.objectives import A2CLoss, ClipPPOLoss, DQNLoss, ReinforceLoss
from training.torchrl_env import make_env
env = make_env(0)
batch = env.reset()
print("ready", tuple(batch.get("observation").shape))
"""
subprocess.check_call([sys.executable, "-u", "-c", check], cwd=dest)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if (root / "main.py").exists():
    os.chdir(root)
subprocess.check_call([sys.executable, "-u", "main.py", "--baselines"], cwd=Path.cwd())
subprocess.check_call([sys.executable, "-u", "main.py", "--screenshot"], cwd=Path.cwd())
print("screenshot", Path("assets/environment.png").resolve(), flush=True)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if (root / "main.py").exists():
    os.chdir(root)

def run(args):
    process = subprocess.Popen(
        [sys.executable, "-u", *args],
        cwd=Path.cwd(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise RuntimeError(f"{' '.join(args)} failed with exit status {code}")

print("10 settings each for DQN, REINFORCE, PPO, and A2C. 20000 frames each.", flush=True)
print("A finished setting is skipped. Progress is copied to Drive after each one.", flush=True)
run(["main.py", "--sweep", "all", "--device", "cpu", "--frames", "20000", "--eval-every", "5000"])


In [ ]:
import shutil
import zipfile
from pathlib import Path

root = Path("/content/Rl_Mission-based_Summative")
if not (root / "main.py").exists():
    root = Path.cwd()
archive = Path("/content/farumasi_rl_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
    for folder in ("models", "logs", "assets"):
        base = root / folder
        if not base.exists():
            continue
        for path in base.rglob("*"):
            if path.is_file() and path.name != ".gitkeep":
                handle.write(path, path.relative_to(root).as_posix())
backup = Path("/content/drive/MyDrive/farumasi_rl")
if backup.exists():
    shutil.copy(archive, backup / archive.name)
    print("copied zip to Drive", flush=True)
print("saved", archive, "bytes", archive.stat().st_size, flush=True)
try:
    from google.colab import files
    files.download(str(archive))
except Exception as error:
    print(error)
